In [ ]:
import numpy as np
from scipy.io import loadmat
import matplotlib.pyplot as plt
import seaborn as sns
import os

%matplotlib widget

In [ ]:

def load_cell_metrics(cell_metrics_file, **kwargs):
    """
    Load and parse a CellExplorer `cell_metrics.cellinfo.mat` file.

    Parameters
    ----------
    cell_metrics_file : str
        Full path to the `.cell_metrics.cellinfo.mat` file
    
    **kwargs : dict, optional
        Additional options for loading and filtering:
        - verbose (bool): If True, print a summary of the loaded data.

    Returns
    -------
    dict
        Dictionary containing:
            - cellIDs : array-like
                CellExplorer cell IDs (1-indexed).
            - cluIDs : array-like
                Kilosort/Phy cluster IDs (0-indexed).
            - tags : dict
                Mapping of brain region name -> list of cellIDs in region.
            - spikeCount : array-like
                Total spike count per cell (indexed by cellID order).
            - putativeCellType : array-like
                Putative cell type labels per cell. (indexed by cellID order)
            - spikes : list of arrays
                Spike time arrays per cell. (indexed by cellID order)
            - shankID : array-like
                Shank ID per cell. (indexed by cellID order)
            - refractoryPeriodViolations : array-like
                Percentage of refractory period violations per cell. (indexed by cellID order)
            - SleepState : dict or None
                If present, a dictionary with keys for the sleep states and values of Nx2 double arrays of [start_time, end_time] for each detected episode of that state.

    Notes
    -----
    - `cellIDs` are 1-indexed (CellExplorer convention).
    - `cluIDs` are 0-indexed (Kilosort convention).
    - Returned arrays preserve ordering from the .mat file.
    """

    # unpacking kwargs
    verbose = kwargs.get("verbose", False)


    if not os.path.exists(cell_metrics_file):
        raise FileNotFoundError(f"Cell metrics file not found: {cell_metrics_file}")

    # Load .mat file
    raw = loadmat(cell_metrics_file, struct_as_record=False, squeeze_me=True)

    if "cell_metrics" not in raw:
        raise KeyError("`cell_metrics` structure not found in .mat file.")

    cell_metrics = mat_struct_to_dict(raw["cell_metrics"])

    # Extract fields (safely)
    cellIDs = cell_metrics.get("cellID")
    cluIDs = cell_metrics.get("cluID")
    spikeCount = cell_metrics.get("spikeCount")
    putativeCellType = cell_metrics.get("putativeCellType")
    shankID = cell_metrics.get("shankID")
    refractoryPeriodViolations = cell_metrics.get("refractoryPeriodViolation")
    general = cell_metrics.get("general")
    states = mat_struct_to_dict(general).get("states") if general is not None else None
    SleepState = mat_struct_to_dict(mat_struct_to_dict(states).get("SleepState")) if states is not None else None


    tags_raw = cell_metrics.get("tags")
    tags = mat_struct_to_dict(tags_raw) if tags_raw is not None else {}

    # extract spike times if available
    spikes_raw = cell_metrics.get("spikes")
    spikes = None
    if spikes_raw is not None:
        spikes_dict = mat_struct_to_dict(spikes_raw)
        spikes = spikes_dict.get("times")




    # ----------------------------
    # Verbose summary (clean + structured)
    # ----------------------------
    if verbose:
        n_cells = len(cellIDs) if cellIDs is not None else 0

        print("\n" + "=" * 60)
        print("Cell Metrics Summary")
        print("=" * 60)
        print(f"File: {cell_metrics_file}")
        print(f"Total cells: {n_cells}")

        # Brain regions
        if tags:
            print("\nBrain regions (tags):")
            for region, ids in tags.items():
                try:
                    count = len(ids)
                except TypeError:
                    count = 1
                print(f"  - {region:<15} : {count:>4} cells")
        else:
            print("\nBrain regions: None")

        # Sleep States
        if SleepState is not None:
            print("\nSleep states:")
            detected_states = SleepState.keys()
            for state in detected_states:
                print(f'    - {state}: {SleepState[state].shape}')
        else:
            print("\nSleep states: None")

        # Cell types
        if putativeCellType is not None:
            unique_types, counts = np.unique(putativeCellType, return_counts=True)
            print("\nPutative cell types:")
            for t, c in zip(unique_types, counts):
                print(f"  - {str(t):<20} : {c:>4} cells")
        else:
            print("\nPutative cell types: None")

        # Shank distribution
        if shankID is not None:
            unique_shanks, counts = np.unique(shankID, return_counts=True)
            print("\nShank distribution:")
            for s, c in zip(unique_shanks, counts):
                print(f"  - Shank {s:<5} : {c:>4} cells")

        print("=" * 60 + "\n")

    return {
        "cellIDs": cellIDs,
        "cluIDs": cluIDs,
        "tags": tags,
        "spikeCount": spikeCount,
        "refractoryPeriodViolations": refractoryPeriodViolations,
        "putativeCellType": putativeCellType,
        "spikes": spikes,
        "shankID": shankID,
        "SleepState": SleepState,
    }


def get_presence_ratio(
    est_counts_per_bin: np.ndarray,
    time_bins: np.ndarray,
    n_coarse_bins: int
) -> np.ndarray:
    """
    Compute presence ratio per unit using coarse time bins.
    A unit is 'present' in a coarse bin if its estimated spike count sum in that bin > 0.
    Vectorized via a (n_time x n_coarse_bins) binning matrix.
    """
    if time_bins.ndim != 1:
        raise ValueError("time_bins must be 1D (monotonic increasing).")
    if est_counts_per_bin.shape[1] != time_bins.size:
        raise ValueError("est_counts_per_bin columns must match len(time_bins).")

    edges = np.linspace(time_bins[0], time_bins[-1], n_coarse_bins + 1)
    # Map each fine time bin into a coarse bin index [0, n_coarse_bins-1]
    bin_idx = np.digitize(time_bins, edges, right=False) - 1
    bin_idx = np.clip(bin_idx, 0, n_coarse_bins - 1)

    # Build a (n_time, n_coarse_bins) one-hot binning matrix (uint8 to save memory)
    n_time = time_bins.size
    B = np.zeros((n_time, n_coarse_bins), dtype=np.uint8)
    B[np.arange(n_time), bin_idx] = 1

    # Sum counts within each coarse bin for every unit: (n_units x n_time) @ (n_time x n_bins)
    coarse_sums = est_counts_per_bin @ B  # shape: (n_units, n_coarse_bins)

    # Presence if sum>0 in a coarse bin; ratio across bins
    presence = (coarse_sums > 0).mean(axis=1)
    return presence


def compute_spike_counts(
    spike_times: list[np.ndarray],
    spike_clusters: np.ndarray,
    window_size: float = 1.0,
    step_size: float = 1.0,
    sigma: float = 0,
    zscore: bool = False,
):
    """
    Compute spike counts using a sliding window approach from Cell Metrics data.
    
    This function processes spike times and cluster assignments from Kilosort/Phy2 which have been loaded from Cell Metrics, and computes spike counts within overlapping sliding windows.
    Optionally, Gaussian smoothing and z-scoring can be applied across time for each unit.

    Parameters
    ----------
    spike_times : list[np.ndarray]
        List of arrays of spike times for each unit.
    spike_clusters : np.ndarray
        Array of cluster or cell IDs corresponding to each spike time.
    window_size : float, optional
        Size of the sliding window in seconds, default is 1.0.
    step_size : float, optional
        Step size between successive windows in seconds, default is 1.0 (non-overlapping
    use_units : str, optional
        Filter for unit types to include:
        - 'all': Include all units
        - 'good': Include only good units
        - 'mua': Include only multi-unit activity
        - 'good/mua': Include both good units and multi-unit activity
        - 'noise': Include only noise units
        Default is 'all'.
    sigma : float, optional
        Standard deviation (in window steps) for Gaussian smoothing kernel. If 0 or None,
        no smoothing is applied. Default is 2.5.
    zscore : bool, optional
        Whether to z-score the spike counts over time for each unit, default is True.
    adj : str or None, optional
        Suffix for spike_times file (e.g., '_sec_adj'), consistent with your existing code.

    Returns
    -------
    spike_count_matrix : ndarray
        Matrix of spike counts (shape: num_units × num_windows).
    time_bins : ndarray
        Array of starting times for each window (seconds).
    units : ndarray
        Array of unit IDs corresponding to rows of `spike_count_matrix`.

    Notes
    -----
    - Counts are raw spike counts per window before optional smoothing/z-scoring.
      To get rates later, you can divide by `window_size`.
    """


    # Return early if no spikes
    if spike_times.size == 0:
        return (
            np.zeros((0, 0), dtype=np.float64),
            np.zeros((0,), dtype=np.float64),
            np.array([], dtype=int),
        )

    # Total duration of recording
    recording_duration = float(spike_times.max())
    if recording_duration < window_size:
        # No full window fits; return empty with units list
        units = np.unique(spike_clusters)
        return (
            np.zeros((len(units), 0), dtype=np.float64),
            np.zeros((0,), dtype=np.float64),
            units,
        )

    # Number of windows and their start times
    num_windows = 1 + int(np.floor((recording_duration - window_size) / step_size))
    time_bins = np.arange(num_windows, dtype=np.float64) * step_size  # window starts

    # Assign each spike to a window start index
    start_idx = np.floor(spike_times / step_size).astype(np.int64)
    valid = (start_idx >= 0) & (start_idx < num_windows)
    start_idx = start_idx[valid]
    spike_times_v = spike_times[valid]
    spike_clusters_v = spike_clusters[valid]

    # Guard against spikes that land in a start bin whose window would end before the spike
    win_end = (start_idx * step_size) + window_size
    valid2 = spike_times_v < win_end
    start_idx = start_idx[valid2]
    spike_clusters_v = spike_clusters_v[valid2]

    # Map units to row indices
    units = np.unique(spike_clusters_v)  # actual units present post-filter
    unit_to_row = {u: i for i, u in enumerate(units)}
    rows = np.fromiter(
        (unit_to_row[u] for u in spike_clusters_v),
        dtype=np.int64,
        count=spike_clusters_v.size,
    )

    # Accumulate counts: (unit, window) -> spike count
    spike_count_matrix = np.zeros((units.size, num_windows), dtype=np.float64)
    np.add.at(spike_count_matrix, (rows, start_idx), 1.0)

    # Optional smoothing (on counts)
    if sigma and sigma > 0:
        from scipy.ndimage import gaussian_filter1d
        for r in range(spike_count_matrix.shape[0]):
            spike_count_matrix[r, :] = gaussian_filter1d(
                spike_count_matrix[r, :],
                sigma=sigma,
                mode='nearest',
            )

    # Optional z-score per unit
    if zscore:
        mean = spike_count_matrix.mean(axis=1, keepdims=True)
        std = spike_count_matrix.std(axis=1, keepdims=True)
        std[std == 0] = 1.0
        spike_count_matrix = (spike_count_matrix - mean) / std

    return spike_count_matrix, time_bins, units

def mat_struct_to_dict(s):
    # for scipy with struct_as_record=False, squeeze_me=True
    return {name: getattr(s, name) for name in getattr(s, "_fieldnames", [])}

In [ ]:
# ----------------------------- Config ---------------------------------

# Data paths and session info
BASEPATH = '/mnt/home/srafilson/ceph/Bilat_R02_20251106'  # base path to the session data (should contain .cell_metrics.cellinfo.mat and .session.mat files)
FIG_DIR = os.path.join(BASEPATH, "seq-hmm-figs")
ANALYSIS_DIR = os.path.join(BASEPATH, "seq-hmm-analysis-results")

# probes, brain regions, and cell types
PROBES = [2, 3]                         # Probes to include in the analysis. There should be a corresponding basePath/baseName_imec{probe}.session.mat for each probe number
BRAIN_REGIONS = ['CA1']                 # Brain regions to include in the analysis (these are stored as tags in Cell Metrics, e.g., 'CA1', 'CA3', 'DG')
NEURON_TYPES = ['Pyramidal Cell', 'Narrow Interneuron', 'Wide Interneuron']       #, 'Narrow Interneuron', 'Wide Interneuron']       # Neuron types to include in the analysis
EXCLUSION_TAGS = ['Bad']                # Tags for excluding neurons (e.g., 'Bad', 'Noise')


# Params for binning spike counts
WINDOW_SIZE = 0.02                      # s
STEP_SIZE   = 0.02                      # s
REFRACTORY_PERIOD_VIOLATIONS = 2        # percentage of allowed ISI less than 2ms

# Params for population burst detection
BURST_THRESHOLD = 3                 # z-scored population firing rate threshold for burst detection
BURST_BOUNDARY_THRESHOLD = 0        # z-scored population firing rate threshold for defining burst boundaries (start and end)
MIN_BURST_DURATION = 0.04           # minimum burst duration in seconds
MIN_INTERBURST_INTERVAL = 0.       # minimum interval between bursts in seconds (will merge bursts that are closer than this)

# Params for the place fields
N_POS_BINS = 50



# Filtering thresholds
MIN_TOTAL_SPIKES   = 500                # minimum total number of spikes across the session
MIN_MEAN_RATE_HZ   = 0.001               # minimum mean firing rate across the session (in Hz) (0.1 noam)
MAX_MEAN_RATE_HZ   = 50                  # maximum mean firing rate across the session (in Hz)
MIN_PRESENCE_RATIO = 0.25               # minimum fraction of time bins in which the neuron is active (spike count > 0)
N_TIME_BINS        = 100               # bins for presence ratio


# defining the hemisphere dictionary. Keys are probe numbers, values are 'L' or 'R' for left/right hemisphere
HEMISPHERE_DICT = {0: 'L', 1: 'L', 2: 'R', 3: 'R'}


# Create figure directory and analysis directory if they don't exist
BASE_NAME = os.path.basename(BASEPATH)
os.makedirs(FIG_DIR, exist_ok=True)
#os.makedirs(ANALYSIS_DIR, exist_ok=True)

In [ ]:
# ------------------------ Load the data from cell metrics file ------------------------
data_dict = {}
for probe in PROBES:
    cell_metrics_file = os.path.join(BASEPATH, f'{BASE_NAME}_imec{probe}.cell_metrics.cellinfo_spikes_adjusted.mat')
    data_dict[probe] = load_cell_metrics(cell_metrics_file, verbose=True)

In [ ]:
# ------------------------ Compute spike counts and presence ratio and filter units ------------------------
shortest_time_bins = np.inf
for probe in PROBES:

    # loading spike times and cluster IDs
    spikes = data_dict[probe]['spikes'] # list of arrays, one per unit, with the spike times for that unit. These are in the order of CellIDs (which are 1-indexed, so spikes[0] corresponds to CellID 1, etc.)
    spike_clusters = np.asarray(data_dict[probe]['cellIDs']) # array of CellIDs corresponding to the spike times. The length of this array should match the length of the spikes list, and the order should correspond (i.e., spike_clusters[i] is the CellID (i+1) for the spike times in spikes[i])

    # Concatenate all spike times and corresponding cluster IDs into flat arrays
    all_spike_times = np.concatenate([np.asarray(s).ravel() for s in spikes]) if spikes is not None else np.array([])
    all_clusters = np.repeat(spike_clusters, [len(s) for s in spikes]) if spikes is not None else np.array([])


    if spikes is None or spike_clusters is None:
        print(f"Probe {probe}: No spike data found. Skipping.")
        continue

    # Compute spike counts per window. Units are a subset of CellIDs the are active in the session and populate the spike count matrix
    spike_count_matrix, time_bins, units = compute_spike_counts(
        spike_times=all_spike_times,
        spike_clusters=all_clusters,
        window_size=WINDOW_SIZE,
        step_size=STEP_SIZE,
        sigma=0,
        zscore=False,
    )

    # Compute presence ratio per unit
    presence_ratio = get_presence_ratio(
        est_counts_per_bin=spike_count_matrix,
        time_bins=time_bins,
        n_coarse_bins=N_TIME_BINS
    )

    # filter units based on presence ratio and total spike count
    total_spikes_per_unit = np.array([len(spikes[u-1]) for u in units])
    valid_units = (total_spikes_per_unit >= MIN_TOTAL_SPIKES) & (presence_ratio >= MIN_PRESENCE_RATIO)
    valid_units &= (total_spikes_per_unit / (time_bins[-1] + STEP_SIZE) >= MIN_MEAN_RATE_HZ)  # mean rate filter
    valid_units &= (total_spikes_per_unit / (time_bins[-1] + STEP_SIZE) <= MAX_MEAN_RATE_HZ)  # max mean rate filter

    # filter units based on brain region
    region_tags = data_dict[probe]['tags']
    if region_tags:
        region_mask = np.zeros_like(units, dtype=bool)
        for region in BRAIN_REGIONS:
            if region in region_tags:
                ids = np.atleast_1d(region_tags[region])
                region_mask |= np.isin(units, ids)
        valid_units &= region_mask
    else:
        print(f"Probe {probe}: No region tags found for filtering.")

    # filtering units based on neuron type
    putativeCellType = data_dict[probe]['putativeCellType']
    if putativeCellType is not None:
        type_mask = np.zeros_like(units, dtype=bool)
        for neuron_type in NEURON_TYPES:
            type_mask |= (putativeCellType[units - 1] == neuron_type)  # units are 1-indexed CellIDs, putativeCellType is in CellID order
        valid_units &= type_mask
    else:
        print(f"Probe {probe}: No putativeCellType found for filtering.")

    
        
    # filter out bad tags
    bad_unit_ids = set()
    tags = data_dict[probe]['tags']
    if tags:
        for bad_tag in EXCLUSION_TAGS:
            if bad_tag in tags:
                ids = np.atleast_1d(tags[bad_tag])
                bad_unit_ids.update(ids.tolist())
        valid_units &= ~np.isin(units, list(bad_unit_ids))
    else:
        print(f"Probe {probe}: No tags found for filtering.")


    # filtering out units with too many refractory period violations
    refractory_violations = data_dict[probe].get('refractoryPeriodViolations') # these are in the full CellExplorer row order
    if refractory_violations is not None:
        refractory_violations = np.asarray(refractory_violations)
        if refractory_violations.size == len(data_dict[probe]['cellIDs']):
            valid_units &= (refractory_violations[units - 1] <= REFRACTORY_PERIOD_VIOLATIONS)
        else:
            print(f"Warning: Length of refractoryPeriodViolations does not match number of cells for probe {probe}. Skipping refractory period filtering.")

    print(f"Probe {probe}: {valid_units.sum()} valid units out of {len(units)} total units after filtering.")

    # Store results back in the data dictionary
    data_dict[probe]['spike_count_matrix'] = spike_count_matrix
    data_dict[probe]['time_bins'] = time_bins
    data_dict[probe]['units'] = units                       # This units array should be used for indexing into the spike_count_matrix
    data_dict[probe]['presence_ratio'] = presence_ratio
    data_dict[probe]['valid_units'] = valid_units           # boolean mask of valid units after filtering to use for the decoding analysis
    data_dict[probe]['bad_unit_indices'] = bad_unit_ids     # these are just the CellIDs that were tagged as bad

    if len(time_bins) < shortest_time_bins:
        shortest_time_bins = len(time_bins)

# clip to the shortest time bins across probes (this is just to make sure all probes have the same time bins for the decoding analysis, since we will concatenate across probes)
for probe in PROBES:
    if 'time_bins' in data_dict[probe]:
        data_dict[probe]['time_bins'] = data_dict[probe]['time_bins'][:shortest_time_bins]
    if 'spike_count_matrix' in data_dict[probe]:
        data_dict[probe]['spike_count_matrix'] = data_dict[probe]['spike_count_matrix'][:, :shortest_time_bins]
    
    print(f"Probe {probe}: time bins clipped to {len(data_dict[probe]['time_bins'])} bins.")

In [ ]:
# concatenate spike count matrices and time bins across probes for the decoding analysis
all_spike_counts = []
all_time_bins = []
for probe in PROBES:
    if 'spike_count_matrix' in data_dict[probe] and 'time_bins' in data_dict[probe]:
        all_spike_counts.append(data_dict[probe]['spike_count_matrix'])
        all_time_bins.append(data_dict[probe]['time_bins'])
    else:
        print(f"Probe {probe}: Missing spike count matrix or time bins for concatenation.")

spike_count_matrix = np.concatenate(all_spike_counts, axis=0) if all_spike_counts else np.array([])
time_bins = all_time_bins[0] if all_time_bins else np.array([])

print(spike_count_matrix.shape, time_bins.shape)


In [ ]:



# plot the spike count matrix
plt.figure(figsize=(10, 6))
sns.heatmap(spike_count_matrix, cmap='viridis', cbar=True)
plt.xlabel('Time bins')
plt.ylabel('Units')
plt.title('Spike Count Matrix (units x time bins)')
plt.tight_layout()
plt.show()


